### Import modules

In [129]:
from pathlib import Path
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.problems.functional import FunctionalProblem
from pymoo.optimize import minimize
from pymoo.core.crossover import Crossover
from pymoo.core.mutation import Mutation
from pymoo.core.repair import Repair
from pymoo.core.duplicate import ElementwiseDuplicateElimination
from pymoo.operators.selection.tournament import TournamentSelection

import pandas as pd
import numpy as np

### Preliminary: Nodes name and Profit score

In [130]:
DIR = Path("../data")

# dataframe of .csv
poi = pd.read_csv(DIR / "POI-Edin.csv", sep=";").query("poiID != 26")
cost = pd.read_csv(DIR / "costProfCat-EdinPOI-all.csv", sep=";").query("`from` != 26 and to != 26")
visits = pd.read_csv(DIR / "userVisits-Edin.csv", sep=";")

time_budget = 420

ids = []
for p in poi.poiID:
    if p != 17:
        ids.append(p)
nodes = ids + [17]

nodes_names = poi.set_index("poiID").loc[ids, "poiName"].to_numpy()
profit_score = cost.groupby("to")["profit"].first().loc[ids].to_numpy(float)

#### Mini section: Walking time cost

In [131]:
WALKING_SPEED = 4000 # 4 km/hr -> 4000 m/hr

TRAVEL_TIME_MATRIX = (cost.pivot(index="from", columns="to", values="cost")
                      .reindex(index=nodes, columns=nodes)
                      .fillna(0)
                      .to_numpy() / (WALKING_SPEED / 60))

TRAVEL_TIME_MATRIX

array([[  0.        ,  27.27374209,  10.50611665,  24.68257209,
        195.71648072, 209.04081511,  37.11452051,   9.50161306,
          9.45436436,  20.65937957,  20.50734404,   4.42349021,
         14.06925805,   9.24032932,   6.68850083,   6.17697923,
          3.13963703,  12.75389382,  28.89108219,  55.60654233,
          9.37784898,  17.5527011 ,  24.95924408,  60.15433186,
         25.2722066 ,  12.9145721 ,   5.62843737,   9.65020039],
       [ 27.27374209,   0.        ,  18.81213844,   2.59163637,
        217.29086371, 231.00403433,  17.05740923,  19.42552168,
         17.88260262,  10.20378457,   9.45792001,  31.15439452,
         16.04153695,  36.3340441 ,  22.3687295 ,  21.81958431,
         28.80163218,  15.9126602 ,  38.70778597,  42.89078594,
         33.20399778,  10.93991151,   3.65465706,  82.79836534,
         51.55235   ,  16.17310562,  21.69618507,  17.77672351],
       [ 10.50611665,  18.81213844,   0.        ,  16.34882661,
        206.04216594, 219.41427159,  2

### Preliminary: Visit duration by mean

In [132]:
# Dedicated POI avg visit in each POI
MIN_VISIT_MIN = 15

photo_count = visits["poiID"].value_counts().sort_index()

visits = visits.sort_values(["userID", "dateTaken"])
new_visits = (visits["poiID"] != visits["poiID"].shift()) | (visits["seqID"] != visits["seqID"].shift()) | (visits["userID"] != visits["userID"].shift())
visits["visitID"] = new_visits.cumsum()

visits_duration = visits.groupby("visitID").agg(poiID=("poiID", "first"), seqID=("seqID", "first"),
                                       userID=("userID", "first"), photos=("photoID", "size"),
                                       t_arrive=("dateTaken", "min"), t_depart=("dateTaken", "max"))

visits_duration["raw_min"] = (visits_duration["t_depart"] - visits_duration["t_arrive"]) / 60
visits_duration["minutes"] = visits_duration["raw_min"].clip(lower=MIN_VISIT_MIN)

per_poi = (visits_duration.groupby("poiID")
           .agg(visits=("minutes", "size"), users=("userID", "nunique"), 
                mean_min=("minutes", "mean"), median_min=("minutes", "median"))
           .join(poi.set_index("poiID")[["poiName", "theme"]])
           .join(photo_count.rename("photo")))

visits_by_mean = per_poi["mean_min"].loc[ids].to_numpy(float)

print("array length: ", len(visits_by_mean))

visits_by_mean

array length:  27


array([54.54788442, 28.34719101, 35.84000532, 23.49233333, 15.        ,
       15.        , 26.85915493, 24.33466969, 31.12832431, 21.52446483,
       21.05236111, 34.75043695, 59.37286325, 60.69866071, 23.13259109,
       17.41045752, 24.27529976, 24.50620748, 47.49125683, 40.28401826,
       33.09351852, 18.81270928, 26.18412073, 15.        , 39.01666667,
       39.24833333, 29.54122299])

### Helper function

In [133]:
def total_walking_time(tour: tuple[int, ...]):
    depot = len(TRAVEL_TIME_MATRIX) - 1
    res = 0
    prev_node = depot
    
    for t in tour:
        res += TRAVEL_TIME_MATRIX[prev_node, t]
        prev_node = t
    
    res += TRAVEL_TIME_MATRIX[prev_node, depot]
    return res

def total_visit_time(tour: tuple[int, ...]):
    res = 0
    for t in tour:
        res += visits_by_mean[t]
    return res

def total_tour_time(tour):
    return total_walking_time(tour) + total_visit_time(tour)

weight = profit_score / visits_by_mean
total_weight = sum(weight)
probability = []

for w in weight:
    p = w / total_weight
    probability.append(float(p))

def random_tour(rng: np.random.Generator):
    # time_budget
    tour= []
    
    for j in rng.choice(len(ids), size=len(ids), replace=False, p=probability):
        pois = tour + [int(j)]
        
        if total_tour_time(pois) <= time_budget:
            tour = pois
    
    return tuple(tour)

### Class preparation

In [134]:
class TourCrossover(Crossover):
    def __init__(self, n_parents, n_offsprings, prob, **kwargs):
        super().__init__(n_parents, n_offsprings, prob, **kwargs)
        
    def _do(self, problem, x, *args, **kwargs):
        rng = kwargs.get("random_state") or np.random.default_rng()
        
        _, n_mating, _ = x.shape
        child = np.empty((2, n_mating, 1), dtype=object)
        
        for m in range(n_mating):
            parent_a, parent_b = x[0, m, 0], x[1, m, 0]
            head_a = parent_a[:rng.integers(len(parent_a) + 1)]
            head_b = parent_b[:rng.integers(len(parent_b) + 1)]
            
            tail_a, tail_b = [], []
            for i in parent_b:
                if i not in head_a:
                    tail_a.append(i)

            for j in parent_a:
                if j not in head_b:
                    tail_b.append(j)
            
            child[0, m, 0] = head_a + tuple(tail_a)
            child[1, m, 0] = head_b + tuple(tail_b)
            
        return child

class TourMutation(Mutation):
    def __init__(self, prob):
        super().__init__(prob)
    
    def _do(self, problem, x, **kwargs):
        rng = kwargs.get("random_state") or np.random.default_rng()
        child = x.copy()
        
        for i in range(len(x)):
            tour = list(x[i, 0])
            op = rng.integers(3) if tour else 0
            
            res = tour.copy()
            match op:
                case 0:
                    not_in_tour = []
                    for poi in range(len(ids)):
                        if poi not in tour:
                            not_in_tour.append(poi)    
                    
                    if not_in_tour:
                        res.insert(rng.integers(len(res) + 1), int(rng.choice(not_in_tour)))
                case 1:
                    res.pop(rng.integers(len(res)))
                case 2:
                    if len(res) > 1:
                        j, k = sorted(rng.choice(len(res), 2, replace=False))
                        if rng.random() < 0.5:
                            res[j], res[k] = res[k], res[j]
                        else:
                            res[j:k + 1] = res[j:k + 1][::-1]
                
            child[i, 0] = tuple(res)
        return child

class RepairTour(Repair):
    def _do(self, problem, x, *args, **kwargs):
        for i in range(len(x)):
            raw_tour = list(x[i, 0])
            tour = []
            
            for rt in raw_tour:
                if rt not in tour:
                    tour.append(rt)
            
            # remove pois from tour with excess time budget 
            while total_tour_time(tour) > time_budget:
                tour_duration = total_tour_time(tour)
                ratio = []
                
                for j, k in enumerate(tour):
                    r = profit_score[k] / (tour_duration - total_tour_time(tour[:j] + tour[j+1:]))
                    ratio.append(r)
                
                tour.pop(int(np.argmin(ratio)))
            
            # filling the tour to meet budget
            poi_not_visited = []
            for idx in range(len(ids)):
                if idx not in tour:
                    poi_not_visited.append(idx)
            
            poi_not_visited = sorted(poi_not_visited, key=lambda p: probability[p], reverse=True)
            
            for idx in poi_not_visited:
                poi = tour + [int(idx)]
                
                if total_tour_time(poi) <= time_budget:
                    tour = poi
            
            x[i, 0] = tuple(tour)
        return x

### Genetic Algorithm method

In [135]:
def objective(x):
    # finding maximum z
    z = 0
    for i in x[0]:
        z += profit_score[i]
    return -z

def time_constraints(x):
    t = total_tour_time(x[0]) - time_budget
    return t

def best_in_tournament(population, bracket, **kwargs):
    # finding minimized z for maximized z objective
    min_z = population.get("F")[:, 0]
    
    res = []
    for competitor in bracket:
        best = min(competitor, key=lambda i: min_z[i])
        res.append(best)
    
    return np.array(res)

def is_same_tour(a, b):
    return a.X[0] == b.X[0]

same_tour = ElementwiseDuplicateElimination(cmp_func=is_same_tour)
problem = FunctionalProblem(1, [objective], constr_ieq=[time_constraints], vtype=object)

def ga_make(N, pc, pm, seed):
    rng = np.random.default_rng(seed)
    start = np.empty((N, 1), dtype=object)
    
    for i in range(N):
        start[i, 0] = random_tour(rng)
    
    algorithm = GA(
        pop_size=N,
        n_offsprings=N,
        sampling=start,
        selection=TournamentSelection(func_comp=best_in_tournament, pressure=3),
        crossover=TourCrossover(2, 2, pc),
        mutation=TourMutation(pm),
        repair=RepairTour(),
        eliminate_duplicates=same_tour,
    )
    return algorithm

### 1. Solve default (50,000 evaluations with 100 stalls cutoff)

Result runtime (jupyter cell execution): 26 minutes 47.3 seconds

max z = 25711

In [136]:
hyperparam_pop_size = [50, 100, 200]
hyperparam_crossover = [0.7, 0.8, 0.9]
hyperparam_mutation = [0.1, 0.2, 0.3]

"""
def run_default(n, pc, pm, seed, max_evals=50_000):
    res = minimize(problem, ga_make(n, pc, pm, seed), ("n_eval", max_evals), seed=seed)
    # -res.F[0] is maximized tour's profit score, while res.X[0] is a raw tour indexes combination
    poi_order = []
    for t in res.X[0]:
        poi_order.append(ids[t])
    return -res.F[0], poi_order, res
"""

def run_default(pop_size, crossover_rate, mutation_rate, seed, max_eval=50_000, n_stall=100):
    algo = ga_make(pop_size, crossover_rate, mutation_rate, seed)
    algo.setup(problem, termination=("n_eval", max_eval), seed=seed)
    
    best = np.inf
    stall = 0
    
    while algo.has_next():
        algo.next()
        min_z = algo.opt.get("F").min() # F is -z, which is needed for maximized problem
        
        if min_z < best:
            best = min_z
            stall = 0
        else:
            stall += 1
        
        if stall >= n_stall:
            break
    
    res = algo.result()
    res.algorithm = algo
    
    poi_order = []    
    for t in res.X[0]:
        poi_order.append(ids[t])
    
    return -res.F[0], poi_order, res

rows = []
for s in range(1001, 1011):
    for N in hyperparam_pop_size:
        for pC in hyperparam_crossover:
            for pM in hyperparam_mutation:
                z, tour, res = run_default(N, pC, pM, s)
                tour_durr = total_tour_time(res.X[0])
                rows.append({
                    "seed": s, "N": N, "pc": pC, "pm": pM,
                    "Z": z,
                    "runtime_s": res.exec_time,
                    "n_evals": res.algorithm.evaluator.n_eval,
                    "tour": " -> ".join(str(p) for p in tour),
                    "tour_duration": tour_durr
                })
                print(f"seed {s}, N = {N}, pc = {pC}, pm = {pM}, Z = {z:.0f}, tour duration (minutes): {tour_durr}")
                pd.DataFrame(rows).to_csv("default_run.csv", index=False)

seed 1001, N = 50, pc = 0.7, pm = 0.1, Z = 25305, tour duration (minutes): 415.7961505511908
seed 1001, N = 50, pc = 0.7, pm = 0.2, Z = 25305, tour duration (minutes): 411.44981540614
seed 1001, N = 50, pc = 0.7, pm = 0.3, Z = 25305, tour duration (minutes): 417.744973463057
seed 1001, N = 50, pc = 0.8, pm = 0.1, Z = 25305, tour duration (minutes): 417.32500769719087
seed 1001, N = 50, pc = 0.8, pm = 0.2, Z = 25305, tour duration (minutes): 418.46908398762935
seed 1001, N = 50, pc = 0.8, pm = 0.3, Z = 25305, tour duration (minutes): 419.0537071806398
seed 1001, N = 50, pc = 0.9, pm = 0.1, Z = 25305, tour duration (minutes): 419.0183885113478
seed 1001, N = 50, pc = 0.9, pm = 0.2, Z = 25305, tour duration (minutes): 410.53671354528853
seed 1001, N = 50, pc = 0.9, pm = 0.3, Z = 25305, tour duration (minutes): 412.742115182636
seed 1001, N = 100, pc = 0.7, pm = 0.1, Z = 25305, tour duration (minutes): 415.2245984135746
seed 1001, N = 100, pc = 0.7, pm = 0.2, Z = 25711, tour duration (minu

### 2. Solve within time limit
max z = 25711 (15 sec), runtime 60 min (approx - need to calculate again)

max z = 25711 (5 sec), runtime = 26 min (approx - need to calculate again)

max z = 25711 (1 sec), runtime = 4 min 44.3s

**Conclusion:** If the termination time limit is too short, the chance of finding optimal solution will be lower.

In [141]:
TIME_LIMIT = "00:00:01"
def limited_time_run(N, pc, pm, seed, time_limit):
    algo = ga_make(N, pc, pm, seed)
    algo.setup(problem, termination=("time", time_limit), seed=seed)
    
    while algo.has_next():
        algo.next()
    
    res = algo.result()
    res.algorithm = algo
    
    poi_order = []    
    for t in res.X[0]:
        poi_order.append(ids[t])
        
    return -res.F[0], poi_order, res

rows_time_limit = []
for s in range(1001, 1011):
    for N in hyperparam_pop_size:
        for pC in hyperparam_crossover:
            for pM in hyperparam_mutation:
                z, tour, res = limited_time_run(N, pC, pM, s, TIME_LIMIT)
                tour_durr = total_tour_time(res.X[0])
                rows_time_limit.append({
                    "seed": s, "N": N, "pc": pC, "pm": pM,
                    "Z": z,
                    "runtime_s": res.exec_time,
                    "n_evals": res.algorithm.evaluator.n_eval,
                    "tour": " -> ".join(str(p) for p in tour),
                    "tour_duration": tour_durr
                })
                print(f"seed {s}, N = {N}, pc = {pC}, pm = {pM}, Z = {z:.0f}, tour duration (minutes): {tour_durr}")
                pd.DataFrame(rows_time_limit).to_csv("time_limited_run_1.csv", index=False)

seed 1001, N = 50, pc = 0.7, pm = 0.1, Z = 25305, tour duration (minutes): 415.7961505511908
seed 1001, N = 50, pc = 0.7, pm = 0.2, Z = 25305, tour duration (minutes): 411.44981540614
seed 1001, N = 50, pc = 0.7, pm = 0.3, Z = 25305, tour duration (minutes): 417.744973463057
seed 1001, N = 50, pc = 0.8, pm = 0.1, Z = 25305, tour duration (minutes): 417.32500769719087
seed 1001, N = 50, pc = 0.8, pm = 0.2, Z = 25305, tour duration (minutes): 418.46908398762935
seed 1001, N = 50, pc = 0.8, pm = 0.3, Z = 25305, tour duration (minutes): 419.0537071806398
seed 1001, N = 50, pc = 0.9, pm = 0.1, Z = 25305, tour duration (minutes): 419.0183885113478
seed 1001, N = 50, pc = 0.9, pm = 0.2, Z = 25305, tour duration (minutes): 410.53671354528853
seed 1001, N = 50, pc = 0.9, pm = 0.3, Z = 25305, tour duration (minutes): 412.742115182636
seed 1001, N = 100, pc = 0.7, pm = 0.1, Z = 25305, tour duration (minutes): 415.2245984135746
seed 1001, N = 100, pc = 0.7, pm = 0.2, Z = 25305, tour duration (minu

### 3. Terminate solver improvement threshold (0.1% improvement rate)

In [ ]:
def run_threshold(pop_size, crossover_rate, mutation_rate, seed, max_eval=50_000, improv_threshold=0.01):
    algo = ga_make(pop_size, crossover_rate, mutation_rate, seed)
    algo.setup(problem, termination=("n_eval", max_eval), seed=seed)
    prev_z = 0.0
    count = 0
    
    best_z = []
    while algo.has_next():
        algo.next()
        curr_z = algo.opt.get("F").min() # F is -z, which is needed for maximized problem
        
        best_z.append(curr_z)
        
        improv_rate = (curr_z - prev_z) / prev_z
        
        if improv_rate < improv_threshold and count == 100:
            break
        elif improv_rate < improv_threshold:
            count += 1
        else:
            prev_z = curr_z
        
    res = algo.result()
    res.algorithm = algo
    
    poi_order = []    
    for t in res.X[0]:
        poi_order.append(ids[t])
    
    return -res.F[0], poi_order, res